**What this notebook is, and what it needs**

- It shows the classes of the `flx` package on the example data in `notebooks/example-dataset`: 100 synthetic
  fingerprint images (10 fingers with 10 images each) made with the SFinGe generator, with the minutiae that
  the generator wrote (`.ist` files).
- It is **not** the route for training on your own images. That route is `docs/training-guide.md` with the
  scripts in `scripts/`. For real images the minutiae come from `scripts/extract_minutiae.py` (Step 4 of
  that guide) as `.mnt` files and are read with `FingerFlowMinutiaMapLoader`.
- It needs the training environment of `docs/installation.md` with the optional tools (`jupyter`). Start
  Jupyter in the root directory of the repository or in `notebooks/`.
- It is stored without outputs. Its cells were run once on a CPU for the documents in `docs/`, with `num_epochs`
  lowered from 20 to 2 in both training cells. With 20 epochs each training cell takes much longer and
  writes the model (0.8 GB) after every epoch.
- Each training cell writes about 1.6 GB (`model.pyt` and `best_model.pyt`) into `example-model/` and
  `example-model-loctexminu/` in the root directory of the repository.
- With the default batch size of 64, the 50 training images form one batch. On the CPU-only machine used
  here (15 GB of memory, shared with other jobs) the first training cell was then stopped by the operating
  system for lack of memory, at more than 12 GB. The run was repeated with the two lines that are commented
  out in that cell (`set_training_batch_size(16)`), which worked.
- **The models trained here are not usable.** 50 training images and 20 epochs of one step each show how the
  classes are called, nothing more. The error rates that the cells print mean nothing.

In this tutorial we will learn to:
- Instantiate a DeepPrintExtractor
- Prepare a training dataset
- Train a DeepPrintExtractor

## Instantiate a DeepPrintExtractor

This package implements a number of variants of the DeepPrint architecture. The wrapper class for all these variants is called `DeepPrintExtractor`.
It has a `fit` method to train (and save) the model as well as an `extract` method to extract the DeepPrint features for fingerprint images. 

You can also try to implement your own models, but currently this is not directly supported by the package.

In [ ]:
from flx.data.dataset import IdentifierSet, Identifier
from flx.extractor.fixed_length_extractor import get_DeepPrint_TexMinu, DeepPrintExtractor
from flx.benchmarks.verification import VerificationBenchmark
from flx.benchmarks.biometric_comparison import BiometricComparison

# We will use the example dataset with 10 subjects and 10 impressions per subject
# Split into 5 training subjects and 5 validation subjects
TRAINING_SUBJECTS = list(range(5))       # Subjects 0-4 for training
VALIDATION_SUBJECTS = list(range(5, 10)) # Subjects 5-9 for validation
IMPRESSIONS_PER_SUBJECT = list(range(10))

training_ids: IdentifierSet = IdentifierSet([
    Identifier(s, i) for s in TRAINING_SUBJECTS for i in IMPRESSIONS_PER_SUBJECT
])
validation_ids: IdentifierSet = IdentifierSet([
    Identifier(s, i) for s in VALIDATION_SUBJECTS for i in IMPRESSIONS_PER_SUBJECT
])

print(f"Training: {training_ids.num_subjects} subjects, {len(training_ids)} samples")
print(f"Validation: {validation_ids.num_subjects} subjects, {len(validation_ids)} samples")

# Create the extractor (note: num_training_subjects must match training set)
extractor: DeepPrintExtractor = get_DeepPrint_TexMinu(
    num_training_subjects=training_ids.num_subjects, 
    num_dims=256
)

# Create a simple verification benchmark for our small validation set
def create_simple_benchmark(subjects, impressions):
    """Create mated and non-mated comparisons for small datasets."""
    comparisons = []
    
    # Mated comparisons: same subject, different impressions
    for s in subjects:
        for i, imp1 in enumerate(impressions):
            for imp2 in impressions[i+1:]:
                comparisons.append(BiometricComparison(
                    Identifier(s, imp1), Identifier(s, imp2)
                ))
    
    # Non-mated comparisons: different subjects, same impression
    for i, s1 in enumerate(subjects):
        for s2 in subjects[i+1:]:
            for imp in impressions[:2]:  # Use first 2 impressions for non-mated
                comparisons.append(BiometricComparison(
                    Identifier(s1, imp), Identifier(s2, imp)
                ))
    
    return VerificationBenchmark(comparisons)

validation_benchmark = create_simple_benchmark(VALIDATION_SUBJECTS, IMPRESSIONS_PER_SUBJECT)
print(f"Validation benchmark: {len(validation_benchmark._comparisons)} comparisons")

## Training the model

Instantiating the model was easy. To train it, first we will load the training data (see the [data tutorial](./dataset_tutorial.ipynb) for how to implement your own dataset).

Besides the fingerprint images, we also need a mapping from subjects to integer labels (for pytorch). For some variants we also need minutiae data. Here it comes from the `.ist` files that the SFinGe generator wrote (`SFingeMinutiaMapLoader`); for real images it comes from `scripts/extract_minutiae.py` (see `docs/training-guide.md`, Step 4). To see how a more complex dataset can be loaded, have a look at `flx/setup/datasets.py`.

Finally, we call the `fit` method, which trains the model and saves it to the specified path.

### Validation

We use a validation set to compute the **Equal Error Rate (EER)** during training. This is a proper biometric metric that measures how well the model distinguishes between same-subject (mated) and different-subject (non-mated) fingerprint pairs.

The validation benchmark defines which pairs to compare:
- **Mated comparisons**: Same subject, different impressions (should have high similarity)
- **Non-mated comparisons**: Different subjects (should have low similarity)

In [ ]:
import os
from pathlib import Path

import torch 

from flx.data.dataset import *
from flx.data.image_loader import SFingeLoader
from flx.data.minutia_map_loader import SFingeMinutiaMapLoader
from flx.data.label_index import LabelIndex
from flx.data.transformed_image_loader import TransformedImageLoader
from flx.image_processing.binarization import LazilyAllocatedBinarizer
from flx.data.image_helpers import pad_and_resize_to_deepprint_input_size

# The root directory of the repository. This works when Jupyter was started
# in the repository root or in its notebooks/ directory.
REPO = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
assert (REPO / "notebooks" / "example-dataset").is_dir(), (
    f"Start Jupyter in the root directory of the repository or in notebooks/ (it runs in {Path.cwd()})"
)

DATASET_DIR: str = str(REPO / "notebooks" / "example-dataset")
# The embedding generation tutorial loads the model from this directory
MODEL_OUTDIR: str = str(REPO / "example-model")

# We will use the SFingeLoader to load the images from the dataset
image_loader = TransformedImageLoader(
        images=SFingeLoader(DATASET_DIR),
        poses=None,
        transforms=[
            LazilyAllocatedBinarizer(5.0),
            pad_and_resize_to_deepprint_input_size,
        ],
    )

# Training datasets
image_dataset = Dataset(image_loader, training_ids)
label_dataset = Dataset(LabelIndex(training_ids), training_ids)
minutia_maps_dataset = Dataset(SFingeMinutiaMapLoader(DATASET_DIR), training_ids)

# Validation dataset (uses same image_loader, different identifiers)
validation_fingerprints = Dataset(image_loader, validation_ids)

# With the default batch size of 64 the 50 training images form one batch,
# which needed more than 12 GB of memory on a CPU. To use less memory,
# remove the "#" from the next two lines (the batch is then 16 images):
# from flx.models.torch_helpers import set_training_batch_size
# set_training_batch_size(16)

print("=" * 60)
print("Training TexMinu")
print("=" * 60)

extractor.fit(
    fingerprints=image_dataset,
    minutia_maps=minutia_maps_dataset,
    labels=label_dataset,
    validation_fingerprints=validation_fingerprints,
    validation_benchmark=validation_benchmark,
    num_epochs=20,
    out_dir=MODEL_OUTDIR
)

print("\nTexMinu training complete!")
print(f"Model saved to: {MODEL_OUTDIR}")

## Training with Localization Network (LocTexMinu)

The **LocTexMinu** variant adds a **localization network**: a small network in front of the model that proposes a rotation and a shift for every image. The image is resampled with them before the rest of the model sees it (a learned alignment).

| Variant | Texture Branch | Minutia Branch | Localization Network |
|---------|----------------|----------------|----------------------|
| TexMinu | ✅ | ✅ | ❌ |
| LocTexMinu | ✅ | ✅ | ✅ |

Whether the localization network makes matching more accurate has to be measured on your data. In the short runs made for the documents of this repository it stayed at "no rotation, no shift" and did not bring differently placed copies of a print closer together. See "Learned alignment" in `docs/tuning.md` and in `docs/measurements.md`.

In [ ]:
from flx.extractor.fixed_length_extractor import get_DeepPrint_LocTexMinu

print("=" * 60)
print("Training LocTexMinu (with Localization Network)")
print("=" * 60)

# LocTexMinu adds a localization network for learned alignment
extractor_loc: DeepPrintExtractor = get_DeepPrint_LocTexMinu(
    num_training_subjects=training_ids.num_subjects, 
    num_dims=256
)

# Use a separate output directory to avoid overwriting the TexMinu model
MODEL_OUTDIR_LOC: str = str(REPO / "example-model-loctexminu")

# Training uses the same datasets as TexMinu - including validation
extractor_loc.fit(
    fingerprints=image_dataset,
    minutia_maps=minutia_maps_dataset,
    labels=label_dataset,
    validation_fingerprints=validation_fingerprints,
    validation_benchmark=validation_benchmark,
    num_epochs=20,
    out_dir=MODEL_OUTDIR_LOC
)

print("\nLocTexMinu training complete!")
print(f"Model saved to: {MODEL_OUTDIR_LOC}")

In [ ]:
import torch
from flx.models.torch_helpers import get_device

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Current device: {get_device()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")